In [ ]:
import sys
import cv2 as cv
import numpy as np
import tensorflow as tf
from PyQt5.QtWidgets import *

# EfficientNet 전용 모듈 및 전처리 함수 로드
from tensorflow.keras.applications.efficientnet import (
    EfficientNetB0,
    decode_predictions,
    preprocess_input,
)
#winsound일때 예외처리
try:
    import winsound
except ImportError:
    winsound = None

# efficientnet 모델에서 imagenet 가중치 설정
model = EfficientNetB0(weights='imagenet')


class VehicleRecognitionApp(QMainWindow):

    def __init__(self):
        super().__init__()
        # 윈도우 창 띄우기
        self.setWindowTitle('AI 교통수단 & 탈것 인식기')
        self.setGeometry(200, 200, 700, 100)
        #버튼 설정
        fileButton = QPushButton('탈것 사진 열기', self)
        recognitionButton = QPushButton('교통수단 인식', self)
        quitButton = QPushButton('나가기', self)
        #버튼 위치 및 크기 설정
        fileButton.setGeometry(10, 10, 100, 30)
        recognitionButton.setGeometry(110, 10, 100, 30)
        quitButton.setGeometry(510, 10, 100, 30)
        #해당 버튼들 클릭했을때 선언해놨던 함수 실행하도록 설정
        fileButton.clicked.connect(self.pictureOpenFunction)
        recognitionButton.clicked.connect(self.recognitionFunction)
        quitButton.clicked.connect(self.quitFunction)

        self.img = None
    #사진 선택
    def pictureOpenFunction(self):
        fname = QFileDialog.getOpenFileName(self, '탈것 사진 선택', './')

        if not fname[0]:
            return

        # 한글 경로 지원 (np.fromfile + cv.imdecode)
        img_array = np.fromfile(fname[0], np.uint8)
        self.img = cv.imdecode(img_array, cv.IMREAD_COLOR)

        if self.img is None:
            QMessageBox.warning(
                self, '경고', '이미지 파일을 읽을 수 없습니다.'
            )
            return
        #이미지 출력
        cv.imshow('Vehicle Image', self.img)
    #사진 인식
    def recognitionFunction(self):
        #self.img에 아무것도 없으면 먼저 사진 선택하라고 경고 박스 띄우고 돌아가기
        if self.img is None:
            QMessageBox.warning(
                self, '경고', '먼저 탈것/교통수단 사진을 열어주세요.'
            )
            return

        # opencv가 bgr로 되어있어서 텐서플로용  rgb로 바꿔주기
        # effic
        img_rgb = cv.cvtColor(self.img, cv.COLOR_BGR2RGB)
        #efficientnetB0에 맞춰서 224* 224 해상도로 바꿔줌
        resized_img = cv.resize(img_rgb, (224, 224))

        # 배치차원 추가한 후에 모델 전처리 적용
        x = np.expand_dims(resized_img, axis=0).astype(np.float32)
        #스케일링 전처리
        x = preprocess_input(x)

        # 사전 훈련된 모델에 사진 넣어서 예측값 도출
        preds = model.predict(x)
        #위에서 3개 출력
        top3 = decode_predictions(preds, top=3)[0]

        img_draw = self.img.copy()

        # 결과 출력하기
        for i in range(3):
            class_name = top3[i][1]  # 위에 3개 이름 저장
            prob = top3[i][2] * 100  # 확률 100분률로 해서 저장

            # 이름 출력
            clean_name = class_name.replace('_', ' ').title()
            text = f'{i+1}위: {clean_name} ({prob:.1f}%)'

            # 글자 모양 세팅
            cv.putText(
                img_draw,
                text,
                (15, 50 + i * 40),
                cv.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 0),
                4,
            )
            #글자 넣기
            cv.putText(
                img_draw,
                text,
                (15, 50 + i * 40),
                cv.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2,
            )
        cv.imshow('Vehicle Image', img_draw)
        if winsound:
            winsound.Beep(1000, 500)
    #함수종료
    def quitFunction(self):
        cv.destroyAllWindows()
        self.close()


app = QApplication(sys.argv)
win = VehicleRecognitionApp()
win.show()
app.exec_()

In [ ]:
import sys
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PyQt5.QtWidgets import *

# EfficientNet 전용 모듈 및 전처리 함수 로드
from tensorflow.keras.applications.efficientnet import (
    EfficientNetB0,
    decode_predictions,
    preprocess_input,
)
#winsound일때 예외처리
try:
    import winsound
except ImportError:
    winsound = None

# efficientnet 모델에서 imagenet 가중치 설정후 로드
model = EfficientNetB0(weights='imagenet')


class VehicleRecognitionApp(QMainWindow):

#생성자 선언
    def __init__(self):
        super().__init__()
        #메인창 제목 및 창 위치와 크기 설정
        self.setWindowTitle('AI 교통수단 & 탈것 인식기')
        self.setGeometry(200, 200, 700, 100)
        #버튼 이름 및 생성
        fileButton = QPushButton('탈것 사진 열기', self)
        recognitionButton = QPushButton('교통수단 인식', self)
        quitButton = QPushButton('나가기', self)
        #각 버트들 위치와 크기 설정
        fileButton.setGeometry(10, 10, 100, 30)
        recognitionButton.setGeometry(110, 10, 100, 30)
        quitButton.setGeometry(510, 10, 100, 30)
        #버튼 클릭했을때 실행할 함수 연결
        fileButton.clicked.connect(self.pictureOpenFunction)
        recognitionButton.clicked.connect(self.recognitionFunction)
        quitButton.clicked.connect(self.quitFunction)
        #초기 이미지 없음
        self.img = None
        #5장 plot할때 사용할 count 변수 선언
        self.count = 0

        # Matplotlib 대화형 모드(plt.ion) 활성화
        plt.ion()
        self.fig = None

    def pictureOpenFunction(self):
        fname = QFileDialog.getOpenFileName(self, '탈것 사진 선택', './')

        if not fname[0]:
            return

        # 한글 경로 지원 (np.fromfile + cv.imdecode)
        img_array = np.fromfile(fname[0], np.uint8)
        self.img = cv.imdecode(img_array, cv.IMREAD_COLOR)
        #이미지 불러왔는데 이미지가 없을때
        if self.img is None:
            QMessageBox.warning(
                self, '경고', '이미지 파일을 읽을 수 없습니다.'
            )
            return
        #이미지가 제대로 불러왔을때 바로 교통수단 인식 버튼 누르게 글 출력
        QMessageBox.information(
            self,
            '안내',
            '이미지가 로드되었습니다. [교통수단 인식] 버튼을 눌러주세요.',
        )
    # 인식 버튼 눌렀을때 self.img가 none이면 경고 박스 출력
    def recognitionFunction(self):
        if self.img is None:
            QMessageBox.warning(
                self, '경고', '먼저 탈것/교통수단 사진을 열어주세요.'
            )
            return

        # opencv 기본 brg을 텐서플로우용 rgb로 바꾸고 efficientnet에 맞춰서 224, 224 크기로 바꿔줌
        img_rgb = cv.cvtColor(self.img, cv.COLOR_BGR2RGB)
        resized_img = cv.resize(img_rgb, (224, 224))

        # 배치 차원 추가민 efficientB0 전처리
        x = np.expand_dims(resized_img, axis=0).astype(np.float32)
        x = preprocess_input(x)

        # 모델에 이미지 넣고 예상 도출 상위 3개만 따로 저장
        preds = model.predict(x)
        top3 = decode_predictions(preds, top=3)[0]

        # 데이터 정리 (상위 3개 클래스 이름 & 확률)
        class_names = [top3[i][1].replace('_', ' ').title() for i in range(3)]
        probs = [top3[i][2] * 100 for i in range(3)]

        # Matplotlib 출력 제어 (누적 출력)
        self.count += 1

        # 첫번째 인식일 경우 Figure 생성
        if self.fig is None or not plt.fignum_exists(self.fig.number):
            self.fig = plt.figure(figsize=(10, 4 * self.count))
        else:
            # 기존 창 크기를 아래로 확장
            self.fig.set_size_inches(10, 4 * self.count)

        # 서브플롯 추가
        # 이미지 출력 (왼쪽)
        ax_img = self.fig.add_subplot(self.count, 2, (self.count - 1) * 2 + 1)
        ax_img.imshow(img_rgb)
        ax_img.set_title(f'Test #{self.count} Image')
        ax_img.axis('off')

        # 확률 막대그래프 출력 (오른쪽)
        ax_bar = self.fig.add_subplot(self.count, 2, (self.count - 1) * 2 + 2)
        bars = ax_bar.barh(class_names[::-1], probs[::-1], color='skyblue')
        ax_bar.set_xlim(0, 100)
        ax_bar.set_xlabel('Probability (%)')
        ax_bar.set_title(f'Top-3 Predictions (#{self.count})')

        # 수치 텍스트 표시
        for bar in bars:
            width = bar.get_width()
            ax_bar.text(
                width + 1,
                bar.get_y() + bar.get_height() / 2,
                f'{width:.1f}%',
                va='center',
            )

        plt.tight_layout()
        plt.draw()
        plt.pause(0.1)

        if winsound:
            winsound.Beep(1000, 500)

    def quitFunction(self):
        plt.close('all')
        cv.destroyAllWindows()
        self.close()


app = QApplication(sys.argv)
win = VehicleRecognitionApp()
win.show()
app.exec_()

In [ ]:
import os
import urllib.request
import pandas as pd

# 1. 폴더 구조 생성
base_dir = "data"
logo_dir = os.path.join(base_dir, "logos")
os.makedirs(logo_dir, exist_ok=True)

# 2. 커피 프로필 데이터셋 생성
data = [
    (1, '스타벅스', 1, 5, 5, 3, 2, 'starbucks.co.kr'),
    (2, '메가커피', 2, 4, 4, 4, 3, 'megagcoffee.com'),
    (3, '투썸플레이스', 2, 4, 4, 4, 3, 'atwosomeplace.co.kr'),
    (4, '빽다방', 2, 3, 4, 4, 3, 'paikdabang.com'),
    (5, '컴포즈커피', 2, 4, 4, 4, 3, 'composecoffee.com'),
    (6, '이디야커피', 2, 3, 3, 3, 4, 'ediya.com'),
    (7, '할리스', 2, 4, 4, 4, 3, 'hollys.co.kr'),
    (8, '엔제리너스', 3, 3, 3, 3, 4, 'angelinus.com'),
    (9, '폴바셋', 3, 4, 3, 4, 4, 'baristapaulbassett.co.kr'),
    (10, '블루보틀', 4, 3, 2, 3, 4, 'bluebottlecoffee.com'),
    (11, '탐앤탐스', 1, 4, 5, 3, 2, 'tomntoms.com'),
    (12, '파스쿠찌', 1, 5, 5, 3, 3, 'caffe-pascucci.co.kr'),
    (13, '더리터', 2, 4, 4, 4, 3, 'theliter.co.kr'),
    (14, '매머드커피', 2, 2, 3, 3, 5, 'mmthcoffee.com'),
    (15, '하삼동커피', 2, 4, 4, 4, 3, 'hasamdong.co.kr'),
    (16, '감성커피', 2, 3, 4, 3, 3, 'gamsungcoffee.com'),
    (17, '커피빈', 2, 4, 4, 4, 4, 'coffeebeankorea.com'),
    (18, '드롭탑', 2, 4, 4, 3, 3, 'cafedroptop.com'),
    (19, '텐퍼센트커피', 2, 4, 3, 4, 4, 'bkkorea.com'),
    (20, '아마스빈', 2, 2, 3, 3, 4, 'amasvin.com')
]

columns = ['id', 'brand_name', 'acidity', 'body', 'bitterness', 'nutty', 'cleanliness', 'domain']
df = pd.DataFrame(data, columns=columns)

# CSV 파일 저장
csv_path = os.path.join(base_dir, "coffee_profiles.csv")
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✅ DB 데이터 생성 완료: {csv_path}")

# 3. 각 브랜드 로고(파비콘) 다운로드
print("⏳ 브랜드 로고 이미지 다운로드 중...")
for _, row in df.iterrows():
    brand_id = row['id']
    domain = row['domain']
    logo_url = f"https://www.google.com/s2/favicons?domain={domain}&sz=128"
    save_path = os.path.join("./logo", f"{brand_id}.png")
    
    try:
        urllib.request.urlretrieve(logo_url, save_path)
        print(f"  - {row['brand_name']} 로고 저장 완료 ({save_path})")
    except Exception as e:
        print(f"  - {row['brand_name']} 로고 저장 실패: {e}")

print("\n🎉 데이터 폴더 구축이 완료되었습니다!")

# 9강

In [5]:
import sys
import cv2 as cv
import numpy as np


# 한글 경로 지원을 위한 이미지 읽기 함수
def imread_korean(filename, flags=cv.IMREAD_COLOR):
    try:
        # np.fromfile로 바이너리를 읽어온 후 cv.imdecode로 디코딩
        img_array = np.fromfile(filename, dtype=np.uint8)
        img = cv.imdecode(img_array, flags)
        return img
    except Exception as e:
        print(f'이미지 로드 중 오류 발생: {e}')
        return None


def construct_yolo_v3():
    f = open('./ch9/coco_names.txt', 'r', encoding='utf-8')
    class_names = [line.strip() for line in f.readlines()]
    f.close()

    # cv.dnn.readNet 대신 cv.dnn.readNetFromDarknet 사용
    #cfg 파일은 레이어 구성 네트워크 파일 weights는 가중치 파라미터 파일
    model = cv.dnn.readNet('./ch9/yolov3.weights', './ch9/yolov3.cfg')

    layer_names = model.getLayerNames()
    unconnected_layers = model.getUnconnectedOutLayers()

    if len(unconnected_layers.shape) == 2:
        out_layers = [layer_names[i[0] - 1] for i in unconnected_layers]
    else:
        out_layers = [layer_names[i - 1] for i in unconnected_layers]

    return model, out_layers, class_names


def yolo_detect(img, yolo_model, out_layers):
    height, width = img.shape[0], img.shape[1]

    # YOLOv3 입력 스케일링 (1.0 / 255.0) 적용
    test_img = cv.dnn.blobFromImage(
        img, 1.0 / 255.0, (448, 448), (0, 0, 0), swapRB=True
    )

    yolo_model.setInput(test_img)
    output3 = yolo_model.forward(out_layers)

    box, conf, id = [], [], []  # 박스, 신뢰도, 부류 번호
    for output in output3:
        for vec85 in output:
            scores = vec85[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.5:  # 신뢰도가 50% 이상인 경우만 취함 (만약 너무 못잡으면 내려서 뭐라도 잡아야한다)
                centerx, centery = int(vec85[0] * width), int(
                    vec85[1] * height
                )
                w, h = int(vec85[2] * width), int(vec85[3] * height)
                x, y = int(centerx - w / 2), int(centery - h / 2)
                box.append([x, y, w, h])  # NMSBoxes 규격에 맞게 [x, y, w, h]로 수정
                conf.append(float(confidence))
                id.append(class_id)

    # NMS 적용
    ind = cv.dnn.NMSBoxes(box, conf, 0.5, 0.4)

    objects = []
    if len(ind) > 0:
        # OpenCV 버전에 따라 1차원 또는 2차원 배열로 반환되는 형태 보정
        for i in ind.flatten():
            x, y, w, h = box[i]
            objects.append([x, y, x + w, y + h, conf[i], id[i]])

    return objects


model, out_layers, class_names = construct_yolo_v3()  # YOLO 모델 생성
colors = np.random.uniform(
    0, 255, size=(len(class_names), 3)
)  # 부류마다 다른 색상 생성

# 한글 경로 지원 함수 사용
img_path = './ch9/soccer.jpg'  # 'D:/한글폴더/축구.jpg' 와 같이 한글 경로 지정 가능
img = imread_korean(img_path)

if img is None:
    sys.exit(f'파일을 찾을 수 없거나 읽을 수 없습니다: {img_path}')

res = yolo_detect(img, model, out_layers)  # YOLO 모델로 물체 검출

for i in range(len(res)):  # 검출된 물체를 영상에 표시
    x1, y1, x2, y2, confidence, class_id = res[i]
    text = f'{class_names[class_id]} {confidence:.3f}'

    color = [int(c) for c in colors[class_id]]
    cv.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv.putText(
        img, text, (x1, max(y1 - 10, 15)), cv.FONT_HERSHEY_PLAIN, 1.5, color, 2
    )

cv.imshow('Object detection by YOLO v.3', img)

cv.waitKey(0)
cv.destroyAllWindows()

In [7]:
import numpy as np
import cv2 as cv
import sys

def construct_yolo_v3():
    f=open('./ch9/coco_names.txt', 'r')
    class_names=[line.strip() for line in f.readlines()]

    model=cv.dnn.readNet('./ch9/yolov3.weights','./ch9/yolov3.cfg')
    layer_names=model.getLayerNames()
    out_layers=[layer_names[i-1] for i in model.getUnconnectedOutLayers()]
    
    return model,out_layers,class_names

def yolo_detect(img,yolo_model,out_layers):
    height,width=img.shape[0],img.shape[1]
    test_img=cv.dnn.blobFromImage(img,1.0/256,(448,448),(0,0,0),swapRB=True)
    
    yolo_model.setInput(test_img)
    output3=yolo_model.forward(out_layers)
    
    box,conf,id=[],[],[]		# 박스, 신뢰도, 부류 번호
    for output in output3:
        for vec85 in output:
            scores=vec85[5:]
            class_id=np.argmax(scores)
            confidence=scores[class_id]
            if confidence>0.5:	# 신뢰도가 50% 이상인 경우만 취함
                centerx,centery=int(vec85[0]*width),int(vec85[1]*height)
                w,h=int(vec85[2]*width),int(vec85[3]*height)
                x,y=int(centerx-w/2),int(centery-h/2)
                box.append([x,y,x+w,y+h])
                conf.append(float(confidence))
                id.append(class_id)
            
    ind=cv.dnn.NMSBoxes(box,conf,0.5,0.4)
    objects=[box[i]+[conf[i]]+[id[i]] for i in range(len(box)) if i in ind]
    return objects

model,out_layers,class_names=construct_yolo_v3()		# YOLO 모델 생성
colors=np.random.uniform(0,255,size=(len(class_names),3))	# 부류마다 색깔

cap=cv.VideoCapture(0,cv.CAP_DSHOW)
if not cap.isOpened(): sys.exit('카메라 연결 실패')

while True:
    ret,frame=cap.read()
    if not ret: sys.exit('프레임 획득에 실패하여 루프를 나갑니다.')
        
    res=yolo_detect(frame,model,out_layers)   
 
    for i in range(len(res)):
        x1,y1,x2,y2,confidence,id=res[i]
        text=str(class_names[id])+'%.3f'%confidence
        cv.rectangle(frame,(x1,y1),(x2,y2),colors[id],2)
        cv.putText(frame,text,(x1,y1+30),cv.FONT_HERSHEY_PLAIN,1.5,colors[id],2)
    
    cv.imshow("Object detection from video by YOLO v.3",frame)
    
    key=cv.waitKey(1) 
    if key==ord('q'): break 
    
cap.release()		# 카메라와 연결을 끊음
cv.destroyAllWindows()

In [13]:
import sys
import cv2 as cv
import numpy as np
import torch


# 1. YOLOv5 모델 로드 (자동으로 최적화된 사전 학습 가중치 다운로드)
def load_yolov5():
    # 'yolov5s' (small), 'yolov5m' (medium), 'yolov5l' (large) 중 선택 가능
    model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
    model.conf = 0.5  # 신뢰도 임계값 (0.5 이상만 감지)
    return model


model = load_yolov5()
class_names = model.names  # 클래스 이름 리스트 추출
colors = np.random.uniform(0, 255, size=(len(class_names), 3))

# 2. 웹캠 연결
cap = cv.VideoCapture(0, cv.CAP_DSHOW)
if not cap.isOpened():
    sys.exit('카메라 연결 실패')

while True:
    ret, frame = cap.read()
    if not ret:
        sys.exit('프레임 획득 실패')

    # 3. YOLOv5 추론 실행 (RGB 이미지 전달)
    rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    results = model(rgb_frame)

    # 4. 검출 결과 추출 [x1, y1, x2, y2, confidence, class_id]
    detections = results.xyxy[0].cpu().numpy()

    # 5. 시각화
    for det in detections:
        x1, y1, x2, y2, confidence, class_id = det
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        class_id = int(class_id)

        text = f'{class_names[class_id]} {confidence:.3f}'
        color = [int(c) for c in colors[class_id]]

        cv.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv.putText(
            frame,
            text,
            (x1, max(y1 - 10, 15)),
            cv.FONT_HERSHEY_PLAIN,
            1.5,
            color,
            2,
        )

    cv.imshow('Object detection from video by YOLOv5', frame)

    key = cv.waitKey(1)
    if key == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

YOLOv5  2026-9-7 Python-3.12.9 torch-2.13.0+cpu CPU

Fusing layers... 


Using cache found in C:\Users\smart/.cache\torch\hub\ultralytics_yolov5_master


YOLOv5s summary: 124 layers, 7,225,885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [14]:
import numpy as np
import cv2 as cv
import sys

def construct_yolo_v3():
    f=open('./ch9/coco_names.txt', 'r')
    class_names=[line.strip() for line in f.readlines()]

    model=cv.dnn.readNet('./ch9/yolov3.weights','./ch9/yolov3.cfg')
    layer_names=model.getLayerNames()
    out_layers=[layer_names[i-1] for i in model.getUnconnectedOutLayers()]
    
    return model,out_layers,class_names

def yolo_detect(img,yolo_model,out_layers):
    height,width=img.shape[0],img.shape[1]
    test_img=cv.dnn.blobFromImage(img,1.0/256,(448,448),(0,0,0),swapRB=True)
    
    yolo_model.setInput(test_img)
    output3=yolo_model.forward(out_layers)
    
    box,conf,id=[],[],[]		# 박스, 신뢰도, 부류 번호
    for output in output3:
        for vec85 in output:
            scores=vec85[5:]
            class_id=np.argmax(scores)
            confidence=scores[class_id]
            if confidence>0.5:	# 신뢰도가 50% 이상인 경우만 취함
                centerx,centery=int(vec85[0]*width),int(vec85[1]*height)
                w,h=int(vec85[2]*width),int(vec85[3]*height)
                x,y=int(centerx-w/2),int(centery-h/2)
                box.append([x,y,x+w,y+h])
                conf.append(float(confidence))
                id.append(class_id)
            
    ind=cv.dnn.NMSBoxes(box,conf,0.5,0.4)
    objects=[box[i]+[conf[i]]+[id[i]] for i in range(len(box)) if i in ind]
    return objects

model,out_layers,class_names=construct_yolo_v3()		# YOLO 모델 생성
colors=np.random.uniform(0,255,size=(len(class_names),3))	# 부류마다 색깔

cap=cv.VideoCapture(0,cv.CAP_DSHOW)
if not cap.isOpened(): sys.exit('카메라 연결 실패')

import time

start=time.time()
n_frame=0
while True:
    ret,frame=cap.read()
    if not ret: sys.exit('프레임 획득에 실패하여 루프를 나갑니다.')
        
    res=yolo_detect(frame,model,out_layers)   
 
    for i in range(len(res)):
        x1,y1,x2,y2,confidence,id=res[i]
        text=str(class_names[id])+'%.3f'%confidence
        cv.rectangle(frame,(x1,y1),(x2,y2),colors[id],2)
        cv.putText(frame,text,(x1,y1+30),cv.FONT_HERSHEY_PLAIN,1.5,colors[id],2)
    
    cv.imshow("Object detection from video by YOLO v.3",frame)
    n_frame+=1
    
    key=cv.waitKey(1) 
    if key==ord('q'): break 

end=time.time()
print('처리한 프레임 수=',n_frame,', 경과 시간=',end-start,'\n초당 프레임 수=',n_frame/(end-start))

cap.release()		# 카메라와 연결을 끊음
cv.destroyAllWindows()

처리한 프레임 수= 72 , 경과 시간= 57.077221155166626 
초당 프레임 수= 1.261448937821714
